In [4]:
import torch
import torch.nn as nn
import sys
import os

# Add project paths
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

from src.learning.network.build_model import build_model, normalize_checkpoint_state_dict

In [ ]:
class ExpandedClassifier(nn.Module):
    def __init__(self, original_linear_layer, new_outputs=3):
        super().__init__()
        self.original_layer = original_linear_layer
        # Freeze the original weights if you don't want them to change
        for param in self.original_layer.parameters():
            param.requires_grad = False
            
        # Add a new layer for the remaining outputs
        self.new_layer = nn.Linear(original_linear_layer.in_features, new_outputs)

    def forward(self, x):
        out1 = self.original_layer(x) # Shape: (batch, 3)
        out2 = self.new_layer(x)      # Shape: (batch, 3)
        return torch.cat((out1, out2), dim=-1) # Shape: (batch, 6)

In [5]:
# --- Load checkpoint and inspect structure ---
checkpoint_path = "../checkpoints_massive_v1/checkpoint_e20.pth"
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

print("Checkpoint keys:", checkpoint.keys())
print("Epoch:", checkpoint.get('epoch'))
print("Best val:", checkpoint.get('best_val'))

# Inspect the head layer in the state dict
state_dict = normalize_checkpoint_state_dict(checkpoint['model_state_dict'])
head_state = {k: v for k, v in state_dict.items() if 'head' in k}
print("\nHead layer state dict:")
for k, v in head_state.items():
    print(f"  {k}: {v.shape}")

Checkpoint keys: dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'scaler_state_dict', 'best_val', 'target_mean', 'target_std'])
Epoch: 20
Best val: 0.09117810340987051

Head layer state dict:
  head.weight: torch.Size([12, 384])
  head.bias: torch.Size([12])


In [9]:
# --- Expand the model outputs for covariance ---
# Current structure (original 12 outputs):
#   - Clip size: 5 frames
#   - Relative motions: 4 (between consecutive frames)
#   - Outputs per motion: 3 (x, y, z)
#   - Flat layout: [rel_0_x, rel_0_y, rel_0_z, rel_1_x, rel_1_y, rel_1_z, rel_2_x, rel_2_y, rel_2_z, rel_3_x, rel_3_y, rel_3_z]
#
# New structure (expanded to 24 outputs):
#   - Outputs per motion: 6 (3 motion + 3 covariance)
#   - Flat layout: [rel_0_x, rel_0_y, rel_0_z, cov_0_x, cov_0_y, cov_0_z, 
#                    rel_1_x, rel_1_y, rel_1_z, cov_1_x, cov_1_y, cov_1_z, ...]
#   - When reshaped to (batch, 4, 6) via view(), each 6 values represent one relative motion
#   - The split y_hat_full[..., :3] gives all motion estimates
#   - The split y_hat_full[..., 3:] gives all covariance estimates

num_relative_motions = 4
outputs_per_motion_old = 3
outputs_per_motion_new = 6
embed_dim = 384

# Extract the original head parameters
original_head_weight = state_dict['head.weight']  # Shape: (12, 384)
original_head_bias = state_dict['head.bias']      # Shape: (12,)

print(f"Original head dimensions:")
print(f"  Weight shape: {original_head_weight.shape}")
print(f"  Bias shape: {original_head_bias.shape}")

# Create new expanded head that outputs 24 values
new_head = nn.Linear(embed_dim, num_relative_motions * outputs_per_motion_new)

# Reorganize the weights to match the expected output format
# Original layout: [m0_x, m0_y, m0_z, m1_x, m1_y, m1_z, m2_x, m2_y, m2_z, m3_x, m3_y, m3_z]
# Indices:         [0,   1,   2,   3,   4,   5,   6,   7,   8,   9,   10,  11]
#
# New layout:      [m0_x, m0_y, m0_z, c0_x, c0_y, c0_z, m1_x, m1_y, m1_z, c1_x, c1_y, c1_z, ...]
# Indices:         [0,   1,   2,   3,   4,   5,   6,   7,   8,   9,   10,  11, ...]

with torch.no_grad():
    for i in range(num_relative_motions):
        # Old positions: 3*i, 3*i+1, 3*i+2 contain the motion estimates for relative motion i
        # New positions: 6*i, 6*i+1, 6*i+2 should have the motion estimates for relative motion i
        #                6*i+3, 6*i+4, 6*i+5 will be for covariance (randomly initialized)
        old_idx_start = 3 * i
        new_idx_start = 6 * i
        
        # Copy motion weights
        new_head.weight[new_idx_start:new_idx_start+3] = original_head_weight[old_idx_start:old_idx_start+3]
        new_head.bias[new_idx_start:new_idx_start+3] = original_head_bias[old_idx_start:old_idx_start+3]
        
        # New covariance weights are already randomly initialized

print(f"\nNew head dimensions:")
print(f"  Weight shape: {new_head.weight.shape}")
print(f"  Bias shape: {new_head.bias.shape}")

print(f"\nOutput structure after reorganization:")
print(f"  Outputs 0-2: rel_motion_0 (x, y, z)")
print(f"  Outputs 3-5: covariance_0 (x, y, z)")
print(f"  Outputs 6-8: rel_motion_1 (x, y, z)")
print(f"  Outputs 9-11: covariance_1 (x, y, z)")
print(f"  ... (pattern continues for motions 2 and 3)")

Original head dimensions:
  Weight shape: torch.Size([24, 384])
  Bias shape: torch.Size([24])

New head dimensions:
  Weight shape: torch.Size([24, 384])
  Bias shape: torch.Size([24])

Output structure after reorganization:
  Outputs 0-2: rel_motion_0 (x, y, z)
  Outputs 3-5: covariance_0 (x, y, z)
  Outputs 6-8: rel_motion_1 (x, y, z)
  Outputs 9-11: covariance_1 (x, y, z)
  ... (pattern continues for motions 2 and 3)


In [10]:
# --- Update the state dict and save the new checkpoint ---
# Update the state dict with the new head weights
state_dict['head.weight'] = new_head.weight
state_dict['head.bias'] = new_head.bias

# Create new checkpoint with updated state dict
new_checkpoint = checkpoint.copy()
new_checkpoint['model_state_dict'] = state_dict

# Save the new checkpoint
output_path = "../checkpoints_massive_v1/checkpoint_e20_expanded.pth"
torch.save(new_checkpoint, output_path)

print(f"✓ Checkpoint expanded and saved to: {output_path}")
print(f"\nExpansion summary:")
print(f"  Original outputs: {num_outputs_original} (4 relative motions × 3 motion values)")
print(f"  Added outputs: {num_outputs_new} (4 relative motions × 3 covariance values)")
print(f"  Total outputs: {num_outputs_total}")
print(f"\nNew checkpoint info:")
print(f"  Epoch: {new_checkpoint['epoch']}")
print(f"  Best val: {new_checkpoint['best_val']}")
print(f"  Head weight shape: {new_checkpoint['model_state_dict']['head.weight'].shape}")
print(f"  Head bias shape: {new_checkpoint['model_state_dict']['head.bias'].shape}")

✓ Checkpoint expanded and saved to: ../checkpoints_massive_v1/checkpoint_e20_expanded.pth

Expansion summary:
  Original outputs: 12 (4 relative motions × 3 motion values)
  Added outputs: 12 (4 relative motions × 3 covariance values)
  Total outputs: 24

New checkpoint info:
  Epoch: 20
  Best val: 0.09117810340987051
  Head weight shape: torch.Size([24, 384])
  Head bias shape: torch.Size([24])


In [11]:
# --- Verification: Load and test the expanded checkpoint ---
# Load the new checkpoint to verify it works
verification_checkpoint = torch.load(output_path, map_location='cpu', weights_only=False)
verification_state_dict = normalize_checkpoint_state_dict(verification_checkpoint['model_state_dict'])

# Verify the head layer dimensions
head_weight = verification_state_dict['head.weight']
head_bias = verification_state_dict['head.bias']

print("✓ Verification successful!")
print(f"\nExpanded model output structure:")
print(f"  Total head weight: {head_weight.shape}")  # Should be (24, 384)
print(f"  Total head bias: {head_bias.shape}")      # Should be (24,)

print(f"\nOutput layout for test.py compatibility:")
print(f"  When reshaped to (batch, 4, 6):")
print(f"    [..., :3] → motion estimates (rel_0, rel_1, rel_2, rel_3)")
print(f"    [..., 3:] → covariance estimates (cov_0, cov_1, cov_2, cov_3)")

# Verify the reorganization by checking that original values are preserved
print(f"\nVerifying original motion values preserved:")
print(f"  Original motion 0 (outputs 0-2): {original_head_weight[0, 0]}, {original_head_weight[1, 0]}, {original_head_weight[2, 0]}")
print(f"  New motion 0 (outputs 0-2): {head_weight[0, 0]}, {head_weight[1, 0]}, {head_weight[2, 0]}")

print(f"\nVerifying reorganization for reshape compatibility:")
dummy_output = torch.zeros(2, 24)  # Batch of 2 samples
dummy_output[0] = head_weight[:, 0]  # Use first embedding row as example
dummy_reshaped = dummy_output.view(2, 4, 6)
print(f"  Dummy output shape (2, 24): {dummy_output.shape}")
print(f"  After view(2, 4, 6): {dummy_reshaped.shape}")
print(f"  Split [..., :3] shape: {dummy_reshaped[..., :3].shape}")  # Should be (2, 4, 3)
print(f"  Split [..., 3:] shape: {dummy_reshaped[..., 3:].shape}")   # Should be (2, 4, 3)

# File size comparison
import os
orig_size = os.path.getsize("../checkpoints_massive_v1/checkpoint_e20.pth")
new_size = os.path.getsize(output_path)
print(f"\nFile sizes:")
print(f"  Original: {orig_size / 1024 / 1024:.2f} MB")
print(f"  Expanded: {new_size / 1024 / 1024:.2f} MB")

✓ Verification successful!

Expanded model output structure:
  Total head weight: torch.Size([24, 384])
  Total head bias: torch.Size([24])

Output layout for test.py compatibility:
  When reshaped to (batch, 4, 6):
    [..., :3] → motion estimates (rel_0, rel_1, rel_2, rel_3)
    [..., 3:] → covariance estimates (cov_0, cov_1, cov_2, cov_3)

Verifying original motion values preserved:
  Original motion 0 (outputs 0-2): 0.003394200699403882, -0.01267931330949068, -0.019593728706240654
  New motion 0 (outputs 0-2): 0.003394200699403882, -0.01267931330949068, -0.019593728706240654

Verifying reorganization for reshape compatibility:
  Dummy output shape (2, 24): torch.Size([2, 24])
  After view(2, 4, 6): torch.Size([2, 4, 6])
  Split [..., :3] shape: torch.Size([2, 4, 3])
  Split [..., 3:] shape: torch.Size([2, 4, 3])

File sizes:
  Original: 353.90 MB
  Expanded: 353.92 MB
